# Notebook 02 — Mortality Table Construction

## Building $q_x$ Tables from Raw Experience Data

This notebook walks through the **end-to-end construction of a mortality table** from raw portfolio experience data, a core actuarial task for French life insurers.

### Actuarial Context

A **mortality table** translates observed deaths and exposures into a coherent set of age-specific probabilities. The key steps are:

1. **Raw mortality rates** $\hat{q}_x = D_x / E_x^c$ — the direct estimate from data, subject to sampling noise.
2. **Graduation (smoothing)** — removes random fluctuation while preserving the underlying mortality pattern. We use the **Whittaker-Henderson** method, the standard in French actuarial practice (recommended by the Institut des Actuaires).
3. **Life table derivation** — from graduated $q_x$, compute survivor function $l_x$, deaths $d_x$, person-years $L_x$, and life expectancy $e_x$.
4. **Benchmarking** — compare the experience table against regulatory references (e.g., **TF 00-02** for female annuitants).

### Notation

| Symbol | Definition |
|--------|------------|
| $q_x$ | Probability of dying between exact ages $x$ and $x+1$ |
| $p_x$ | $1 - q_x$, probability of surviving |
| $l_x$ | Number of survivors at age $x$ (radix $l_0 = 100{,}000$) |
| $d_x$ | $l_x - l_{x+1}$, expected deaths between ages $x$ and $x+1$ |
| $e_x$ | Complete life expectancy at age $x$ |
| $\mu_x$ | Force of mortality (continuous hazard rate) |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import poisson
from pathlib import Path
import sys

# Project imports
sys.path.insert(0, str(Path("../src").resolve()))
from mortality_tables import raw_qx, whittaker_henderson_graduation, build_life_table

# Plotting defaults
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11

print("Imports OK")

## 1. Generate Synthetic Experience Data

We simulate a realistic French female annuitant portfolio using the **Gompertz mortality model**:

$$\mu_x = \alpha \cdot e^{\beta x}$$

where $\alpha$ and $\beta$ are calibrated to approximate observed French female mortality.
Deaths are drawn from a Poisson distribution: $D_x \sim \text{Poisson}(E_x \cdot \mu_x)$.

This serves as a **fallback** when confidential portfolio data is not available.

In [ ]:
# --- Gompertz parameters calibrated to French female mortality ---
np.random.seed(42)

ages = np.arange(0, 106)  # ages 0 to 105
n_ages = len(ages)

# Gompertz-Makeham: mu_x = c + alpha * exp(beta * x)
alpha_gomp = 5e-5
beta_gomp = 0.085
c_makeham = 0.0005  # Makeham constant (accidents, background mortality)

mu_true = c_makeham + alpha_gomp * np.exp(beta_gomp * ages)

# Convert force of mortality to qx: qx = 1 - exp(-mu_x)
qx_true = 1 - np.exp(-mu_true)

# Simulate exposures: larger portfolio for working ages, smaller at extremes
base_exposure = 5000
exposure_shape = np.exp(-0.5 * ((ages - 55) / 25) ** 2)  # centered on 55
exposures = (base_exposure * exposure_shape).astype(int)
exposures = np.clip(exposures, 50, base_exposure)  # minimum 50 lives

# Draw deaths from Poisson
deaths = poisson.rvs(mu=exposures * mu_true, random_state=42)
deaths = np.minimum(deaths, exposures)  # cannot exceed exposures

print(f"Age range: {ages[0]}–{ages[-1]}")
print(f"Total exposures: {exposures.sum():,}")
print(f"Total deaths: {deaths.sum():,}")
print(f"Crude overall mortality: {deaths.sum() / exposures.sum():.4f}")

In [ ]:
# Quick look at the data
data_df = pd.DataFrame({
    "age": ages,
    "exposures": exposures,
    "deaths": deaths,
    "qx_true": qx_true
})
data_df.head(10)

## 2. Compute Raw Mortality Rates $\hat{q}_x$

The `raw_qx` function computes:
$$m_x = \frac{D_x}{E_x^c} \quad \Rightarrow \quad \hat{q}_x = \frac{m_x}{1 + 0.5\,m_x}$$

The second formula applies the **Balducci assumption** (uniform distribution of deaths) to convert central death rates $m_x$ into initial rates $q_x$.

In [ ]:
# Compute raw qx from observed deaths and exposures
qx_raw = raw_qx(deaths.astype(float), exposures.astype(float))

print(f"Raw qx at age 0:  {qx_raw[0]:.6f}")
print(f"Raw qx at age 40: {qx_raw[40]:.6f}")
print(f"Raw qx at age 65: {qx_raw[65]:.6f}")
print(f"Raw qx at age 85: {qx_raw[85]:.6f}")
print(f"Raw qx at age 100: {qx_raw[100]:.6f}")

## 3. Whittaker-Henderson Graduation

Raw rates are **noisy**, especially at extreme ages where exposures are small. The Whittaker-Henderson method finds graduated rates $q_x^*$ that minimize:

$$F = h \sum_x (q_x^* - \hat{q}_x)^2 + (1-h) \sum_x (\Delta^z q_x^*)^2$$

- **First term**: fidelity to observed data (weighted by $h$)
- **Second term**: smoothness penalty on $z$-th order differences

With $h \to 1$: graduated = raw. With $h \to 0$: polynomial of degree $z-1$.

Standard practice: $h \in [0.05, 0.3]$, $z = 2$ or $3$.

In [ ]:
# Graduate the raw qx with default parameters (h=0.1, z=2)
qx_graduated = whittaker_henderson_graduation(qx_raw, h=0.1, z=2)

print("Graduated qx sample values:")
for a in [0, 20, 40, 60, 80, 100, 105]:
    print(f"  Age {a:3d}: raw = {qx_raw[a]:.6f}  |  graduated = {qx_graduated[a]:.6f}  |  true = {qx_true[a]:.6f}")

In [ ]:
# --- Plot: Raw vs Graduated qx (log scale) ---
fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(ages, qx_raw, s=12, alpha=0.5, color="steelblue", label="Raw $\\hat{q}_x$ (observed)", zorder=3)
ax.plot(ages, qx_graduated, color="crimson", linewidth=2.2, label="Graduated $q_x^*$ (Whittaker-Henderson)", zorder=4)
ax.plot(ages, qx_true, color="black", linewidth=1.2, linestyle="--", label="True $q_x$ (Gompertz-Makeham)", zorder=2)

ax.set_yscale("log")
ax.set_xlabel("Age")
ax.set_ylabel("$q_x$ (log scale)")
ax.set_title("Raw vs Graduated Mortality Rates — Whittaker-Henderson Smoothing")
ax.legend(loc="upper left", fontsize=10)
ax.set_xlim(0, 105)
ax.set_ylim(1e-5, 1)

plt.tight_layout()
plt.show()

The graduated curve (red) closely tracks the true generating model (dashed) while smoothing out the Poisson noise visible in the raw rates (blue dots). This is especially important at young ages (0-20) and very old ages (90+) where data is sparse.

## 4. Build the Complete Life Table

From graduated $q_x$, we derive the full life table:

| Column | Formula |
|--------|----------|
| $l_x$ | $l_{x+1} = l_x \cdot (1 - q_x)$, with $l_0 = 100{,}000$ |
| $d_x$ | $l_x - l_{x+1}$ |
| $L_x$ | $\frac{1}{2}(l_x + l_{x+1})$ — person-years lived |
| $T_x$ | $\sum_{t=x}^{\omega} L_t$ — total future person-years |
| $e_x$ | $T_x / l_x$ — complete life expectancy |

In [ ]:
# Build the life table from graduated qx
life_table = build_life_table(qx_graduated, age_start=0, radix=100_000)

print(f"Life table: {len(life_table)} rows, ages {life_table['age'].min()}–{life_table['age'].max()}")
print(f"Life expectancy at birth (e0): {life_table.loc[life_table['age'] == 0, 'ex'].values[0]:.2f} years")
print(f"Life expectancy at 65 (e65):   {life_table.loc[life_table['age'] == 65, 'ex'].values[0]:.2f} years")

In [ ]:
# Display selected rows of the life table
display_ages = [0, 1, 5, 10, 20, 30, 40, 50, 60, 65, 70, 75, 80, 85, 90, 95, 100, 105]
display_df = life_table[life_table["age"].isin(display_ages)].copy()
display_df["qx"] = display_df["qx"].map("{:.6f}".format)
display_df["px"] = display_df["px"].map("{:.6f}".format)
display_df["ex"] = display_df["ex"].map("{:.2f}".format)
display_df["lx"] = display_df["lx"].map("{:,.0f}".format)
display_df["dx"] = display_df["dx"].map("{:,.0f}".format)
display_df["Lx"] = display_df["Lx"].map("{:,.0f}".format)
display_df["Tx"] = display_df["Tx"].map("{:,.0f}".format)
display_df = display_df.set_index("age")
display_df

## 5. Comparison with French Reference Table TF 00-02

The **TF 00-02** (Table Feminine 2000-2002) is the regulatory mortality table for female annuitants in France, published by the Institut des Actuaires. It is used as a **minimum basis** for annuity reserving.

Below we construct a synthetic TF 00-02-like reference using published Gompertz parameters and compare it with our experience table.

In [ ]:
# --- Synthetic TF 00-02 reference (approximate Gompertz-Makeham fit) ---
# These parameters approximate published TF 00-02 values
alpha_tf = 3.0e-5
beta_tf = 0.092
c_tf = 0.0003

mu_tf = c_tf + alpha_tf * np.exp(beta_tf * ages)
qx_tf0002 = 1 - np.exp(-mu_tf)
qx_tf0002 = np.clip(qx_tf0002, 0, 1)

# Build life table for TF 00-02
lt_tf = build_life_table(qx_tf0002, age_start=0, radix=100_000)

print(f"TF 00-02 (synthetic) — e0: {lt_tf.loc[lt_tf['age'] == 0, 'ex'].values[0]:.2f} years")
print(f"TF 00-02 (synthetic) — e65: {lt_tf.loc[lt_tf['age'] == 65, 'ex'].values[0]:.2f} years")
print(f"Experience table     — e0: {life_table.loc[life_table['age'] == 0, 'ex'].values[0]:.2f} years")
print(f"Experience table     — e65: {life_table.loc[life_table['age'] == 65, 'ex'].values[0]:.2f} years")

In [ ]:
# --- Plot: qx comparison (experience vs TF 00-02) ---
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(ages, qx_graduated, color="crimson", linewidth=2, label="Experience table (graduated)")
ax.plot(ages, qx_tf0002, color="#2E86C1", linewidth=2, linestyle="--", label="TF 00-02 (synthetic reference)")

ax.set_yscale("log")
ax.set_xlabel("Age")
ax.set_ylabel("$q_x$ (log scale)")
ax.set_title("Experience Table vs French Reference TF 00-02")
ax.legend(fontsize=11)
ax.set_xlim(0, 105)
ax.set_ylim(1e-5, 1)

# Shade region where experience is heavier than reference
ax.fill_between(ages, qx_graduated, qx_tf0002,
                where=qx_graduated > qx_tf0002,
                alpha=0.15, color="red", label="Experience > Reference")
ax.fill_between(ages, qx_graduated, qx_tf0002,
                where=qx_graduated < qx_tf0002,
                alpha=0.15, color="green", label="Experience < Reference")
ax.legend(fontsize=10, loc="upper left")

plt.tight_layout()
plt.show()

## 6. Survivor Curves $l_x$

The survivor function shows the expected number of survivors at each age out of an initial cohort of 100,000.

In [ ]:
# --- Plot: Survivor curves lx ---
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(life_table["age"], life_table["lx"], color="crimson", linewidth=2.2,
        label=f"Experience table ($e_0$={life_table.loc[0, 'ex']:.1f})")
ax.plot(lt_tf["age"], lt_tf["lx"], color="#2E86C1", linewidth=2.2, linestyle="--",
        label=f"TF 00-02 ($e_0$={lt_tf.loc[0, 'ex']:.1f})")

# Mark key ages
for mark_age in [65, 80]:
    lx_exp = life_table.loc[life_table["age"] == mark_age, "lx"].values[0]
    lx_tf = lt_tf.loc[lt_tf["age"] == mark_age, "lx"].values[0]
    ax.axvline(mark_age, color="grey", linestyle=":", alpha=0.5)
    ax.annotate(f"Age {mark_age}", xy=(mark_age, lx_exp), xytext=(mark_age + 2, lx_exp + 3000),
                fontsize=9, color="grey")

ax.set_xlabel("Age")
ax.set_ylabel("$l_x$ (survivors out of 100,000)")
ax.set_title("Survivor Curves — Experience vs TF 00-02")
ax.legend(fontsize=11)
ax.set_xlim(0, 105)
ax.set_ylim(0, 105_000)

plt.tight_layout()
plt.show()

## 7. Life Expectancy $e_x$ by Age

The complete (period) life expectancy at age $x$ represents the average remaining lifetime for a person who has survived to age $x$.

In [ ]:
# --- Plot: Life expectancy ex by age ---
fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(life_table["age"], life_table["ex"], color="crimson", linewidth=2.2,
        label="Experience table")
ax.plot(lt_tf["age"], lt_tf["ex"], color="#2E86C1", linewidth=2.2, linestyle="--",
        label="TF 00-02")

# Annotate key values
for tbl, name, color, offset in [(life_table, "Exp", "crimson", 1.5),
                                   (lt_tf, "TF", "#2E86C1", -2.5)]:
    for mark_age in [0, 65]:
        ex_val = tbl.loc[tbl["age"] == mark_age, "ex"].values[0]
        ax.annotate(f"{name}: $e_{{{mark_age}}}$={ex_val:.1f}",
                    xy=(mark_age, ex_val),
                    xytext=(mark_age + 5, ex_val + offset),
                    fontsize=9, color=color,
                    arrowprops=dict(arrowstyle="->", color=color, lw=0.8))

ax.set_xlabel("Age")
ax.set_ylabel("$e_x$ (years)")
ax.set_title("Complete Life Expectancy by Age")
ax.legend(fontsize=11)
ax.set_xlim(0, 105)
ax.set_ylim(0, None)

plt.tight_layout()
plt.show()

## 8. Sensitivity Analysis: Smoothing Parameter $h$

The parameter $h$ controls the trade-off between **fidelity** and **smoothness**:
- $h$ close to 1: graduated rates stay near the raw data (risk of overfitting)
- $h$ close to 0: very smooth curve (risk of underfitting, losing real features)

We test several values to visualize the effect.

In [ ]:
# --- Sensitivity analysis: varying h ---
h_values = [0.01, 0.05, 0.1, 0.3, 0.5, 0.9]
palette = sns.color_palette("viridis", len(h_values))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left panel: qx curves
ax = axes[0]
ax.scatter(ages, qx_raw, s=8, alpha=0.3, color="grey", label="Raw $\\hat{q}_x$", zorder=1)
ax.plot(ages, qx_true, color="black", linewidth=1, linestyle=":", label="True $q_x$", zorder=2)

for i, h in enumerate(h_values):
    qx_h = whittaker_henderson_graduation(qx_raw, h=h, z=2)
    ax.plot(ages, qx_h, color=palette[i], linewidth=1.5, label=f"$h={h}$", zorder=3)

ax.set_yscale("log")
ax.set_xlabel("Age")
ax.set_ylabel("$q_x$ (log scale)")
ax.set_title("Graduated $q_x$ for Different Smoothing Parameters")
ax.legend(fontsize=8, ncol=2)
ax.set_xlim(0, 105)
ax.set_ylim(1e-5, 1)

# Right panel: resulting e0 and e65
ax2 = axes[1]
e0_vals = []
e65_vals = []
for h in h_values:
    qx_h = whittaker_henderson_graduation(qx_raw, h=h, z=2)
    lt_h = build_life_table(qx_h, age_start=0)
    e0_vals.append(lt_h.loc[lt_h["age"] == 0, "ex"].values[0])
    e65_vals.append(lt_h.loc[lt_h["age"] == 65, "ex"].values[0])

ax2.plot(h_values, e0_vals, "o-", color="crimson", linewidth=2, markersize=7, label="$e_0$ (at birth)")
ax2.plot(h_values, e65_vals, "s-", color="#2E86C1", linewidth=2, markersize=7, label="$e_{65}$")

ax2.set_xlabel("Smoothing parameter $h$")
ax2.set_ylabel("Life expectancy (years)")
ax2.set_title("Impact of $h$ on Life Expectancy Estimates")
ax2.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Summary table of sensitivity results
sensitivity_df = pd.DataFrame({
    "h": h_values,
    "e0": [f"{v:.2f}" for v in e0_vals],
    "e65": [f"{v:.2f}" for v in e65_vals],
    "delta_e0_vs_h01": [f"{v - e0_vals[2]:+.2f}" for v in e0_vals],
    "delta_e65_vs_h01": [f"{v - e65_vals[2]:+.2f}" for v in e65_vals],
})
sensitivity_df.columns = ["h", "e(0)", "e(65)", "delta e(0) vs h=0.1", "delta e(65) vs h=0.1"]
sensitivity_df.set_index("h")

## Key Takeaways

1. **Raw mortality rates** are noisy, especially at extreme ages. Graduation is essential before using rates for reserving or pricing.

2. **Whittaker-Henderson** provides a principled trade-off between fit and smoothness. The parameter $h = 0.1$ (default) works well for typical French portfolio data.

3. **Life expectancy estimates** are moderately sensitive to the choice of $h$ at birth, but more stable at age 65 where data is typically richer.

4. **Comparison with TF 00-02** allows actuaries to assess whether the portfolio experience is lighter or heavier than the regulatory reference — critical for Solvency II best-estimate assumptions.

**Next step**: Notebook 03 applies the **Lee-Carter model** to project these mortality rates forward and quantify longevity risk.